# Fase 4: ¿por qué no sirvieron las medidas?

En la fase 3 pasó algo que no esperábamos. Le dimos a la red 26 medidas de la célula sacadas de
contornos dibujados a mano por especialistas, y no mejoró nada: 0.9719 sin medidas contra 0.9690
con medidas, y la prueba estadística dijo que ni siquiera hay diferencia real.

Un resultado así se puede reportar como "no funcionó" y ya. Pero es mucho mejor explicar **por
qué**, y hay una hipótesis clara que se puede poner a prueba:

> La red ya está calculando esa información por su cuenta, mirando la foto. Dársela otra vez es
> repetirle algo que ya sabe.

La forma de comprobarlo es al revés de lo que uno pensaría. En vez de usar la red para adivinar la
clase, la usamos para adivinar **las medidas**. Cuando la red procesa una foto produce 1280
números que la resumen. Si a partir de esos 1280 números se puede recuperar la proporción
núcleo/célula, entonces esa proporción estaba ahí adentro todo el tiempo.

Eso convierte un resultado nulo en un mecanismo explicado.

## Preparativos

Igual que siempre. Además cargamos las medidas que ya calculamos en la fase 3, así no hay que
recalcularlas.

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
print("Modelo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

In [ ]:
import pandas as pd, numpy as np, kagglehub
from sklearn.model_selection import StratifiedGroupKFold

DEST = "/content/drive/MyDrive/sipakmed"
df = pd.read_csv(f"{DEST}/indice_celulas.csv").reset_index(drop=True)
F  = pd.read_csv(f"{DEST}/medidas_fase3.csv")          # las 26 medidas de la fase 3

path = kagglehub.dataset_download("prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed")
base_viejo = df["img"].iloc[0].split("/im_")[0]
for col in ["img", "cyt", "nuc"]:
    df[col] = df[col].str.replace(base_viejo, path, regex=False)

SEMILLA = 42
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
df["fold_lamina"] = -1
for k, (_, val) in enumerate(sgkf.split(df, df["clase"], groups=df["lamina"])):
    df.loc[df.index[val], "fold_lamina"] = k

CLASES = sorted(df["clase"].unique())
IDX = {c: i for i, c in enumerate(CLASES)}
Y = df["clase"].map(IDX).values

print("Células:", len(df), "| medidas:", F.shape[1])
print("Imágenes encontradas:", df["img"].apply(os.path.exists).all())

## Cargar las imágenes

Lo mismo de las fases anteriores: se decodifican una vez y quedan en memoria.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

print("Cargando imágenes...")
CACHE = np.zeros((len(df), 224, 224, 3), dtype=np.uint8)
for i, ruta in enumerate(df["img"]):
    CACHE[i] = np.array(Image.open(ruta).convert("RGB").resize((224, 224), Image.BILINEAR))
    if i % 500 == 0:
        print(f"  {i}/{len(df)}", end="\r")
print(f"Listo: {CACHE.nbytes/1e6:.0f} MB")

MEDIA = [0.485, 0.456, 0.406]
DESV  = [0.229, 0.224, 0.225]

tf_train = v2.Compose([
    v2.RandomHorizontalFlip(), v2.RandomVerticalFlip(), v2.RandomRotation(20),
    v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
    v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEDIA, DESV),
])
tf_eval = v2.Compose([v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEDIA, DESV)])


class Celulas(Dataset):
    def __init__(self, posiciones, tf):
        self.pos = np.asarray(posiciones)
        self.tf = tf

    def __len__(self):
        return len(self.pos)

    def __getitem__(self, i):
        p = self.pos[i]
        return self.tf(torch.from_numpy(CACHE[p]).permute(2, 0, 1)), Y[p]

## Entrenar y guardar el resumen de cada foto

Volvemos a entrenar la misma red de la fase 3 (solo imagen, sin medidas), pero esta vez, además
de la predicción, nos guardamos los **1280 números** que la red produce antes de decidir la clase.
Ese vector es su resumen interno de la foto.

Un cuidado importante: el resumen de cada célula se saca del modelo **que no la vio durante el
entrenamiento**. Si usáramos un modelo que ya la conoce, el experimento no valdría nada, sería el
mismo error de fondo que venimos evitando desde la fase 1.

In [ ]:
import torch.nn as nn, random, time
from torchvision import models

EPOCAS = 12
LOTE = 64
LR = 3e-4
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def fijar_semilla(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def crear_red():
    base = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    base.classifier = nn.Identity()               # nos quedamos con los 1280 numeros
    cabeza = nn.Sequential(nn.Linear(1280, 256), nn.ReLU(), nn.Dropout(0.3),
                           nn.Linear(256, len(CLASES)))
    return base.to(DEV), cabeza.to(DEV)


EMB = np.zeros((len(df), 1280), dtype=np.float32)   # resumen de cada foto
pred_img = np.full(len(df), -1)

for k in range(5):
    t0 = time.time()
    fijar_semilla(SEMILLA)
    tr = np.where(df["fold_lamina"].values != k)[0]
    te = np.where(df["fold_lamina"].values == k)[0]

    dl_tr = DataLoader(Celulas(tr, tf_train), batch_size=LOTE, shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True)
    dl_te = DataLoader(Celulas(te, tf_eval), batch_size=128, shuffle=False,
                       num_workers=2, pin_memory=True)

    base, cabeza = crear_red()
    params = list(base.parameters()) + list(cabeza.parameters())
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCAS)
    criterio = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda")

    for ep in range(EPOCAS):
        base.train(); cabeza.train()
        for x, y in dl_tr:
            x, y = x.to(DEV), y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = criterio(cabeza(base(x)), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        sched.step()

    # Guardamos resumen y prediccion del pliegue que quedo fuera
    base.eval(); cabeza.eval()
    embs, preds = [], []
    with torch.no_grad():
        for x, y in dl_te:
            with torch.amp.autocast("cuda"):
                z = base(x.to(DEV))
                s = cabeza(z)
            embs.append(z.float().cpu().numpy())
            preds.append(s.argmax(1).cpu().numpy())
    EMB[te] = np.concatenate(embs)
    pred_img[te] = np.concatenate(preds)

    del base, cabeza; torch.cuda.empty_cache()
    print(f"  pliegue {k} listo ({time.time()-t0:.0f}s)")

from sklearn.metrics import f1_score
print("\nmacro F1 de la red:", round(f1_score(Y, pred_img, average="macro"), 4))
np.save("/content/emb_finetune.npy", EMB)

## Un punto de comparación

Para saber si lo que encontremos es mérito del entrenamiento o algo que ya venía de fábrica,
sacamos también el resumen de la **misma red sin entrenar en nuestras células**, tal como viene
preentrenada con fotos genéricas de internet.

Si la red entrenada recupera las medidas mucho mejor que esta, la diferencia es lo que aprendió
específicamente de las células.

In [ ]:
base_gen = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
base_gen.classifier = nn.Identity()
base_gen = base_gen.to(DEV).eval()

EMB_GEN = np.zeros((len(df), 1280), dtype=np.float32)
dl = DataLoader(Celulas(np.arange(len(df)), tf_eval), batch_size=128,
                shuffle=False, num_workers=2, pin_memory=True)

pos = 0
with torch.no_grad():
    for x, _ in dl:
        with torch.amp.autocast("cuda"):
            z = base_gen(x.to(DEV))
        z = z.float().cpu().numpy()
        EMB_GEN[pos:pos+len(z)] = z
        pos += len(z)

del base_gen; torch.cuda.empty_cache()
print("Listo:", EMB_GEN.shape)

## La prueba: recuperar las medidas desde el resumen

Para cada medida entrenamos una regresión sencilla que intenta predecirla a partir de los 1280
números. Se usan los mismos pliegues por lámina de siempre, y el escalado se ajusta solo con el
grupo de entrenamiento.

El resultado se mide con R², que va de 0 a 1:

- **R² cercano a 1:** la medida se recupera casi perfecto, o sea que ya estaba dentro del resumen.
- **R² cercano a 0:** no está, la red no la codifica.

Elegimos doce medidas representativas de cada tipo: tamaño, proporción, forma, color, textura, y
el tamaño original del recorte.

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score

MEDIDAS = ["nc_area", "nuc_area", "cyt_area", "nuc_circ", "nuc_alarg", "nuc_sol",
           "int_nuc", "contraste_nuc_cit", "std_nuc", "glcm_contraste",
           "desplaz_nucleo", "ancho_orig"]

ALFAS = np.logspace(-1, 4, 12)


def sondear(EMBED, nombre_medida):
    """Intenta recuperar una medida a partir del resumen de la red."""
    objetivo = F[nombre_medida].values.astype(float)
    validos = ~np.isnan(objetivo)
    pred = np.full(len(df), np.nan)

    for k in range(5):
        tr = np.where((df["fold_lamina"].values != k) & validos)[0]
        te = np.where((df["fold_lamina"].values == k) & validos)[0]

        mu, sd = EMBED[tr].mean(0), EMBED[tr].std(0) + 1e-6   # escalado solo con train
        modelo = RidgeCV(alphas=ALFAS)
        modelo.fit((EMBED[tr] - mu) / sd, objetivo[tr])
        pred[te] = modelo.predict((EMBED[te] - mu) / sd)

    ok = ~np.isnan(pred)
    return r2_score(objetivo[ok], pred[ok]), pred


filas = []
predicciones = {}
for medida in MEDIDAS:
    r2_ft, p_ft = sondear(EMB, medida)
    r2_gen, _   = sondear(EMB_GEN, medida)
    filas.append({"medida": medida, "R2_red_entrenada": r2_ft, "R2_red_generica": r2_gen})
    predicciones[medida] = p_ft
    print(f"  {medida:20s} entrenada {r2_ft:6.3f} | genérica {r2_gen:6.3f}")

sonda = pd.DataFrame(filas).sort_values("R2_red_entrenada", ascending=False)
sonda.round(3)

## Verlo en un gráfico

In [ ]:
import matplotlib.pyplot as plt

s = sonda.sort_values("R2_red_entrenada")
y = np.arange(len(s))
plt.figure(figsize=(9, 6))
plt.barh(y - 0.2, s["R2_red_entrenada"], height=0.4, label="Red entrenada con células")
plt.barh(y + 0.2, s["R2_red_generica"], height=0.4, label="Red genérica, sin entrenar")
plt.yticks(y, s["medida"])
plt.xlabel("R²: qué tan bien se recupera la medida desde el resumen de la red")
plt.axvline(0, color="k", lw=0.8)
plt.legend(); plt.tight_layout(); plt.show()

Y el caso concreto de la proporción núcleo/célula, que es la medida estrella del diagnóstico y la
que las publicaciones suelen añadir como aporte propio. Cada punto es una célula: en el eje
horizontal su valor real medido del contorno, en el vertical lo que la red deja adivinar.

In [ ]:
real = F["nc_area"].values
est = predicciones["nc_area"]
ok = ~np.isnan(real) & ~np.isnan(est)

plt.figure(figsize=(6, 6))
plt.scatter(real[ok], est[ok], s=6, alpha=0.3)
lim = [0, np.percentile(real[ok], 99.5)]
plt.plot(lim, lim, "r--", lw=1)
plt.xlim(lim); plt.ylim(lim)
plt.xlabel("Proporción real, medida del contorno")
plt.ylabel("Proporción recuperada del resumen de la red")
plt.title(f"R² = {sonda[sonda.medida=='nc_area']['R2_red_entrenada'].values[0]:.3f}")
plt.tight_layout(); plt.show()

## Cómo leer esto

Dos lecturas posibles, y las dos sirven:

- **Las medidas con R² alto ya están dentro de la red.** Dárselas de nuevo no agrega información,
  y eso explica el resultado nulo de la fase 3. Si la proporción núcleo/célula sale con R² alto,
  queda demostrado que la red la calcula sola.
- **Las medidas con R² bajo son información que la red no tiene.** Ahí sí habría algo que ganar.

Presta atención especial a `ancho_orig`. Es el tamaño en píxeles del recorte original, y como
escalamos todo a 224x224 antes de entrenar, esa información fue borrada a propósito. Si su R² sale
bajo mientras el resto sale alto, tenemos identificado exactamente qué es lo único que vale la
pena darle a la red. Y eso se puede probar.

## Probar la única medida que falta

Entrenamos la misma red pero pasándole solo el tamaño original del recorte, dos números, en vez de
las 26 medidas. Comparamos contra el modelo sin medidas de la fase 3.

Si esto mejora aunque las 26 medidas no lo hicieran, el mensaje es fuerte: no se trata de darle
más información, sino de darle **la que efectivamente se perdió en el preprocesamiento**.

In [ ]:
class RedConTamano(nn.Module):
    def __init__(self, n_extra):
        super().__init__()
        base = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        base.classifier = nn.Identity()
        self.base = base
        self.cabeza = nn.Sequential(nn.Linear(1280 + n_extra, 256), nn.ReLU(), nn.Dropout(0.3),
                                    nn.Linear(256, len(CLASES)))

    def forward(self, img, extra):
        return self.cabeza(torch.cat([self.base(img), extra], dim=1))


class CelulasT(Dataset):
    def __init__(self, posiciones, extra, tf):
        self.pos = np.asarray(posiciones)
        self.extra = extra.astype(np.float32)
        self.tf = tf

    def __len__(self):
        return len(self.pos)

    def __getitem__(self, i):
        p = self.pos[i]
        return (self.tf(torch.from_numpy(CACHE[p]).permute(2, 0, 1)),
                torch.from_numpy(self.extra[i]), Y[p])


TAM = F[["ancho_orig", "alto_orig"]]
pred_tam = np.full(len(df), -1)

for k in range(5):
    t0 = time.time()
    fijar_semilla(SEMILLA)
    tr = np.where(df["fold_lamina"].values != k)[0]
    te = np.where(df["fold_lamina"].values == k)[0]

    mu, sd = TAM.iloc[tr].mean(), TAM.iloc[tr].std().replace(0, 1)   # solo con train
    e_tr = ((TAM.iloc[tr] - mu) / sd).values
    e_te = ((TAM.iloc[te] - mu) / sd).values

    dl_tr = DataLoader(CelulasT(tr, e_tr, tf_train), batch_size=LOTE, shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True)
    dl_te = DataLoader(CelulasT(te, e_te, tf_eval), batch_size=128, shuffle=False,
                       num_workers=2, pin_memory=True)

    modelo = RedConTamano(2).to(DEV)
    opt = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCAS)
    criterio = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda")

    for ep in range(EPOCAS):
        modelo.train()
        for img, ex, y in dl_tr:
            img, ex, y = img.to(DEV), ex.to(DEV), y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = criterio(modelo(img, ex), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        sched.step()

    modelo.eval()
    preds = []
    with torch.no_grad():
        for img, ex, y in dl_te:
            with torch.amp.autocast("cuda"):
                s = modelo(img.to(DEV), ex.to(DEV))
            preds.append(s.argmax(1).cpu().numpy())
    pred_tam[te] = np.concatenate(preds)

    del modelo; torch.cuda.empty_cache()
    print(f"  pliegue {k} listo ({time.time()-t0:.0f}s)")

## Comparación final

In [ ]:
from sklearn.metrics import matthews_corrcoef, accuracy_score
from statsmodels.stats.contingency_tables import mcnemar

fase3 = pd.read_csv(f"{DEST}/predicciones_fase3.csv")

tabla = pd.DataFrame([
    {"modelo": "Solo imagen",
     "macro_f1": f1_score(Y, pred_img, average="macro"),
     "mcc": matthews_corrcoef(Y, pred_img)},
    {"modelo": "Imagen + las 26 medidas (fase 3)",
     "macro_f1": f1_score(Y, fase3["pred_fusion"].values, average="macro"),
     "mcc": matthews_corrcoef(Y, fase3["pred_fusion"].values)},
    {"modelo": "Imagen + solo el tamaño original",
     "macro_f1": f1_score(Y, pred_tam, average="macro"),
     "mcc": matthews_corrcoef(Y, pred_tam)},
]).round(4)
print(tabla.to_string(index=False), "\n")

ok_a = (pred_img == Y); ok_b = (pred_tam == Y)
t = [[int((ok_a & ok_b).sum()),  int((ok_a & ~ok_b).sum())],
     [int((~ok_a & ok_b).sum()), int((~ok_a & ~ok_b).sum())]]
print("Solo acierta sin tamaño:", t[0][1], "| solo acierta con tamaño:", t[1][0])
r = mcnemar(t, exact=False, correction=True)
print(f"p = {r.pvalue:.2e} ->", "diferencia real" if r.pvalue < 0.05 else "no hay diferencia clara")

## Guardar

In [ ]:
sonda.to_csv(f"{DEST}/sonda_fase4.csv", index=False)
np.save(f"{DEST}/emb_finetune.npy", EMB)

sal = df[["stem", "clase", "lamina", "fold_lamina"]].copy()
sal["pred_img"] = pred_img
sal["pred_tam"] = pred_tam
sal.to_csv(f"{DEST}/predicciones_fase4.csv", index=False)
print("Guardado en", DEST)

## Resumen de la fase 4

Lo que aporta esta fase al trabajo:

- Convierte el resultado nulo de la fase 3 en una explicación con evidencia. No es que las medidas
  no sirvan, es que la red **ya las tiene calculadas por dentro**, y eso se puede medir.
- Compara contra una red sin entrenar para separar lo que aprendió de las células de lo que ya
  venía de fábrica.
- Identifica qué información sí se pierde en el preprocesamiento, y la prueba directamente en vez
  de quedarse en la especulación.

Es también una crítica concreta a una práctica común: añadir medidas calculadas a mano a una red
profunda suele venderse como aporte, y aquí se muestra con datos que en este problema no agrega
nada, porque la red ya llegó ahí sola.

**Lo que sigue (fase 5):** medir dónde mira la red. Como tenemos los contornos reales del núcleo,
se puede calcular qué porcentaje de su atención cae dentro del núcleo, y ver si los errores
coinciden con mirar en el lugar equivocado. Más las pruebas de aguante ante cambios de color y
desenfoque, que simulan otro laboratorio.